In [8]:
import numpy as np
import matplotlib.pyplot as plt
import os
from tifffile import imread, imwrite
import cv2


In [9]:
def save_fused_tomogram(fused_tomogram, output_folderpath, _skip_preprocessing,
                                      output_voxel_size, parameters, sample_name=''):
    """
    Save the 3D fused tomogram as a TIFF file with correct axis order.

    :param fused_tomogram: numpy array of the fused tomogram (z, y, x)
    :param output_filepath: string, path to save the output TIFF file
    """
    # Ensure the data is in the correct format (z, y, x)
    if len(fused_tomogram.shape) != 3:
        raise ValueError("Fused tomogram must be a 3D numpy array (z, y, x).")

    # create metadata for the TIFF file
    if _skip_preprocessing:
        metadata = {
            'axes': 'ZYX',
            'spacing': parameters['px_size_z'],  # distanza tra slice (Z)
            'unit': 'um',  # unità di misura, può essere anche 'micron'
            'resolution': (1.0 / parameters['px_size_xy'], 1.0 / parameters['px_size_xy']),  # in pixel per µm
        }
    else:
        metadata = {
            'axes': 'ZYX',
            'spacing': output_voxel_size,  # distanza tra slice (Z)
            'unit': 'um',  # unità di misura, può essere anche 'micron'
            'resolution': (1.0 / output_voxel_size, 1.0 / output_voxel_size),  # in pixel per µm
        }

    # Save the 3D array as a TIFF file
    fname = '{}_fused_tomogram.tif'.format(sample_name) if sample_name else 'fused_tomogram.tif'
    output_filepath = os.path.join(output_folderpath, fname)
    imsave(output_filepath, fused_tomogram, imagej=True, metadata=metadata)
    return output_filepath


def plot_matches(img1, img2, kp1, kp2, matches, title='Matches'):
    img_matches = cv2.drawMatches(img1, kp1, img2, kp2, matches, None,
                                   flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
    plt.figure(figsize=(15, 10))
    plt.imshow(img_matches, cmap='gray')
    plt.title(title)
    plt.axis('off')
    plt.show()
    

def estimate_isotropic_scaling_and_translation(img1, img2, _plot_matches_flag=True, plot_inliers_only=True):
    # Rileva feature con ORB
    orb = cv2.ORB_create(1000)
    kp1, des1 = orb.detectAndCompute(img1, None)
    kp2, des2 = orb.detectAndCompute(img2, None)

    # Matcher con ratio test
    bf = cv2.BFMatcher(cv2.NORM_HAMMING)
    raw_matches = bf.knnMatch(des1, des2, k=2)
    good_matches = [m for m, n in raw_matches if m.distance < 0.75 * n.distance]
    print("Number of good matches found: {}".format(len(good_matches)))

    if len(good_matches) < 6:
        raise ValueError("Troppo pochi match buoni ({}) per stimare la trasformazione.".format(len(good_matches)))

    # Ottieni i punti corrispondenti
    pts1 = np.float32([kp1[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    pts2 = np.float32([kp2[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)

    # Stima la trasformazione con RANSAC
    M, inliers = cv2.estimateAffinePartial2D(pts2, pts1, method=cv2.RANSAC, ransacReprojThreshold=3)

    if M is None:
        raise ValueError("Non è stato possibile stimare la trasformazione.")

    # Filtra i match inlier per il plot
    if _plot_matches_flag:
        if plot_inliers_only and inliers is not None:
            inlier_matches = [m for i, m in enumerate(good_matches) if inliers[i]]
            plot_matches(img1, img2, kp1, kp2, inlier_matches, title='Inlier Matches (RANSAC)')
        else:
            plot_matches(img1, img2, kp1, kp2, good_matches, title='Good Matches (ratio test)')

    # Calcola la scala isotropica
    scale_x = np.linalg.norm(M[0, :2])
    scale_y = np.linalg.norm(M[1, :2])
    scale = (scale_x + scale_y) / 2

    # Estrai traslazione
    tx = M[0, 2]
    ty = M[1, 2]

    return scale, tx, ty


def apply_scale_and_translation(img, scale, tx, ty):
    """
    Applica una trasformazione di similitudine (scala isotropica + traslazione) all'immagine.

    Parameters:
        img   : immagine di input (numpy array)
        scale : fattore di scala isotropico
        tx    : traslazione lungo x
        ty    : traslazione lungo y

    Returns:
        Immagine trasformata
    """
    # Costruisci matrice affine: solo scala e traslazione, niente rotazione
    M = np.array([
        [scale, 0, tx],
        [0, scale, ty]
    ], dtype=np.float32)

    # Applica la trasformazione affine
    h, w = img.shape[:2]
    transformed = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR)

    return transformed

In [10]:
from fuse_dual_tomograms_simplified import plot_rgb_frames

In [11]:
# load the selected frame for fusion from hardcoded path
# CAMI heart WGA example
left_path = '/home/fra/ubuntu_data/Dual_MesoSPIM/preprocessing_devel/Samples_test/Cami_heart_WGA/left_z170_ds6um.tif'
right_path = '/home/fra/ubuntu_data/Dual_MesoSPIM/preprocessing_devel/Samples_test/Cami_heart_WGA/right_z170_ds6um.tif'

# N3
# left_path = '/home/fra/ubuntu_data/Dual_MesoSPIM/preprocessing_devel/Samples_test/N3_try_scale/L3_z55_ds6um.tif'
# right_path = '/home/fra/ubuntu_data/Dual_MesoSPIM/preprocessing_devel/Samples_test/N3_try_scale/R3_z55_ds6um.tif'




outpath = os.path.dirname(left_path)

# load with tifffile
left_frame = imread(left_path)
right_frame = imread(right_path)

# check the shape of the tomograms
print(f'Left tomogram shape: {left_frame.shape}')
print(f'Right tomogram shape: {right_frame.shape}')



Left tomogram shape: (1733, 1733)
Right tomogram shape: (1733, 1733)


In [12]:
# plot with plot_rgb_frames
plot_rgb_frames(left_frame, right_frame, title='Alignment Check',
                    red_ch='Left CAM', green_ch='Right CAM', outpath=outpath, _save=True)



Plot saved at: /home/fra/ubuntu_data/Dual_MesoSPIM/preprocessing_devel/Samples_test/Cami_heart_WGA/Alignment_Check_(R:_Left_CAM,_G:_Right_CAM).png


In [13]:
right_scale_xy, right_tx, right_ty = estimate_isotropic_scaling_and_translation(left_frame, right_frame)
print('Estimated scale factor: {}'.format(right_scale_xy))
print('Estimated translation (tx, ty): ({}, {})'.format(right_tx, right_ty))

Number of good matches found: 57
Estimated scale factor: 0.9937152591100413
Estimated translation (tx, ty): (-34.50117587093646, -3.406359635598035)


/tmp/ipykernel_18309/121508364.py:43: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [14]:
# apply the estimated scale and translation to the right frame
right_frame_ok = apply_scale_and_translation(right_frame, right_scale_xy, right_tx, right_ty)

# plot rgb frames with the transformed right frame
plot_rgb_frames(left_frame, right_frame_ok, title='Aligned Frames',
                    red_ch='Left CAM', green_ch='Right CAM (aligned)', outpath=outpath, _save=True)

Plot saved at: /home/fra/ubuntu_data/Dual_MesoSPIM/preprocessing_devel/Samples_test/Cami_heart_WGA/Aligned_Frames_(R:_Left_CAM,_G:_Right_CAM_(aligned)).png


In [15]:
# concatebate frames only to test saveing 2d stack with fiji metadata
aligned_frames = np.concatenate((left_frame[np.newaxis, ...], right_frame_ok[np.newaxis, ...]), axis=0)

# print shape aligned frames
print(f'Aligned frames shape: {aligned_frames.shape}')

Aligned frames shape: (2, 1733, 1733)


In [16]:
def save_tomogram_for_fiji(tomogram, output_folderpath, voxel_size, fname='', axes='ZYX'):
    """
    Salva un tomogramma 3D come file TIFF compatibile con Fiji.

    :param tomogram: numpy array del tomogramma (z, y, x)
    :param output_folderpath: stringa, percorso della cartella di output
    :param voxel_size: dimensione del voxel in micrometri (float o tuple per ZYX)
    :param parameters: dizionario opzionale con parametri aggiuntivi (es. 'px_size_xy', 'px_size_z')
    :param fname: nome del campione per il file di output (opzionale)
    :param axes: stringa che specifica l'ordine degli assi (default: 'ZYX')
    :param dtype: tipo di dato per il salvataggio (default: np.uint16)
    :return: percorso del file salvato
    """
    # Verifica che il tomogramma sia un array 3D
    if len(tomogram.shape) != 3:
        raise ValueError("Il tomogramma deve essere un array numpy 3D (z, y, x).")

    # Determina la dimensione del voxel
    if isinstance(voxel_size, (float, int)):
        voxel_size = (voxel_size, voxel_size, voxel_size)
    elif len(voxel_size) != 3:
        raise ValueError("voxel_size deve essere un singolo valore o una tupla di 3 valori (Z, Y, X).")

    # Salva l'array 3D come file TIFF
    fname = f"{fname}_tomogramZYX.tif" if fname and not fname.endswith('.tif') else (
        fname if fname else "tomogramZYX.tif")
    output_filepath = os.path.join(output_folderpath, fname)

    imwrite(output_filepath,
            tomogram,
            imagej=True,
            metadata={
                'axes': axes,
                'spacing': voxel_size[0],  # distanza tra slice (Z)
                'unit': 'um',  # unità di misura
            },
            resolution=(1.0 / voxel_size[2], 1.0 / voxel_size[1]),  # XY resolution (from ZYX format)
            )
    return output_filepath



def save_fused_tomogram(fused_tomogram, output_folderpath, _skip_preprocessing,
                        output_voxel_size, parameters, sample_name=''):
    """
    Salva il tomogramma fuso utilizzando la funzione save_tomogram_for_fiji.

    :param fused_tomogram: numpy array del tomogramma fuso (z, y, x)
    :param output_folderpath: stringa, percorso della cartella di output
    :param _skip_preprocessing: booleano, indica se saltare il preprocessing
    :param output_voxel_size: dimensione del voxel in micrometri
    :param parameters: dizionario con parametri aggiuntivi
    :param sample_name: nome del campione per il file di output (opzionale)
    :return: percorso del file salvato
    """
    # Determina la dimensione del voxel
    if _skip_preprocessing:
        voxel_size = (parameters['px_size_z'], parameters['px_size_xy'], parameters['px_size_xy'])
    else:
        voxel_size = (output_voxel_size, output_voxel_size, output_voxel_size)

    print("voxel_size for saving: ", voxel_size)

    # Salva il tomogramma fuso come file TIFF compatibile con Fiji
    output_filepath = save_tomogram_for_fiji(
        tomogram=fused_tomogram,
        output_folderpath=output_folderpath,
        voxel_size=voxel_size,
        fname=sample_name,
        axes='ZYX'
    )

    return output_filepath

In [34]:
parameters = {
    'px_size_xy': 3.25,  # um
    'px_size_z': 3.25,   # um
}



(3, 3.25, 3.25)

In [31]:
outpath

'/home/fra/ubuntu_data/Dual_MesoSPIM/preprocessing_devel/Samples_test/Cami_heart_WGA'

In [32]:
# save the aligned frames as a tomogram for fiji

output_filepath = save_tomogram_for_fiji(aligned_frames, outpath, output_voxel_size, fname='aligned_frames', axes='ZYX')